In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
from solver import *

# 2D tests in periodic conditions

## Stationnary equation
### Free of spatial variables

$$\psi\mathrm{Op}(\xi^2 + \eta^2 + 1, u)(x, y) = -\sin x \sin y$$

Tests the 2D stationary pseudo-differential solver (`solve_stationary_psiOp()`) on a constant-symbol elliptic operator $\xi^2 + \eta^2 + 1$ over $[0, 2\pi)^2$ ($N_x = N_y = 512$). Since $(\xi^2 + \eta^2 + 1) \mapsto 1^2 + 1^2 + 1 = 3$ on the mode $\sin x \sin y$, the exact solution is $u(x,y) = -\tfrac{1}{3}\sin x \sin y$.

In [ ]:
# Symbolic variables
x, y, xi, eta = symbols('x y xi eta', real=True)
u = Function('u')(x, y)

# Symbolic equation
symbol = xi**2 + eta**2 + 1
equation = Eq(psiOp(symbol, u), -sin(x) * sin(y))

# Define exact solution
def u_exact(x, y):
    return -np.sin(x) * np.sin(y) / 3

# Solver creation
solver = PDESolver(equation)

# Domain
Lx, Ly = 2 * np.pi, 2 * np.pi
Nx, Ny = 512, 512

# Setup with exact solution as source
solver.setup(Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny, initial_condition=None)

# Stationary solution
u_num = solver.solve_stationary_psiOp()

# Accuracy test
solver.test(u_exact=u_exact, threshold=1e-10, component='real')

solver.show_stationary_solution(u=u_num, component='real')

### Depending on spatial variables

$$\psi\mathrm{Op}(\xi^2 + \eta^2 + x^2 + y^2 + 1, u)(x, y) = (-3x^2 - 3y^2 + 5)\,e^{-(x^2 + y^2)}$$

Tests the 2D stationary `psiOp` solver using a first-order asymptotic inverse (`solve_stationary_psiOp(order=1)`) for the spatially varying symbol $p(x,y,\xi,\eta) = \xi^2 + \eta^2 + x^2 + y^2 + 1$ on $L_x = L_y = 10$ ($N_x = N_y = 64$), verified against the Gaussian exact solution $u(x,y) = e^{-(x^2 + y^2)}$.

In [ ]:
# 1) Define symbolic variables
x, y, xi, eta = symbols('x y xi eta', real=True)
u = Function('u')(x, y)

# 2) Define the 2D symbol p(x, y, xi, eta)
p_symbol = xi**2 + eta**2 + x**2 + y**2 + 1

# 3) Choose an exact solution u_exact(x, y)
u_exact_expr = exp(-x**2 - y**2)

def u_exact(x, y):
    return np.exp(-x**2 - y**2)

# 4) Compute the right-hand side f_expr = P[u_exact] symbolically
f_expr = (-3*x**2 - 3*y**2 + 5)*u_exact_expr

# 5) Build the stationary equation: psiOp(p_symbol, u) = f_expr
equation_2d = Eq(psiOp(p_symbol, u), f_expr)

# 6) Initialize and setup the solver
solver2d = PDESolver(equation_2d)
solver2d.setup(Lx=10.0, Ly=10.0, Nx=64, Ny=64, initial_condition=None)

# 7) Solve stationary psiOp problem (first-order asymptotic inverse)
u_num2d = solver2d.solve_stationary_psiOp(order=1)

# 8) Test numerical vs exact
solver2d.test(u_exact=u_exact, threshold=3e-1, component='real')


## Transport equation

$$u_t = -u_x - u_y, \qquad u(x,y,0) = e^{-(x^2 + y^2)}$$

Tests 2D diagonal advection at unit velocity $(1, 1)$ on $[-5, 5)^2$ ($L_x = L_y = 10$, $N_x = N_y = 512$, $L_t = 3.0$). Starting from a centered 2D Gaussian pulse, the exact solution translates diagonally with periodic wrapping across both boundaries.

In [ ]:
# Definition of the 2D transport equation
t, x, y, kx, ky, omega = symbols('t x y kx ky omega')
u_func = Function('u') 
u = u_func(t, x, y)

transport_eq = Eq(diff(u, t), -diff(u, x) - diff(u, y))  # Diagonal transport

# Creation of the solver with periodic conditions
solver = PDESolver(transport_eq)

# Configuration of the 2D domain and initial condition
L = 10.0  # Square domain [-5, 5] × [-5, 5]
N = 512   # Spatial resolution
Lt=3.0
solver.setup(
    Lx=L, Ly=L,
    Nx=N, Ny=N,
    Lt=Lt, Nt=500,
    initial_condition=lambda x, y: np.exp(-(x**2 + y**2)),  # 2D Gaussian
    n_frames=100

)

# Exact solution with periodic wrapping
def u_exact(x, y, t):
    Lx = Ly = L  # Square domain
    x_shift = (x - t + Lx/2) % Lx - Lx/2  # Periodic wrapping in x
    y_shift = (y - t + Ly/2) % Ly - Ly/2  # Periodic wrapping in y
    return np.exp(-(x_shift**2 + y_shift**2))

# Solving and validation
solver.solve()

# Automatic testing
n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='real') for i in range(n_test + 1)]

## Heat equation
### With PDE

$$u_t = u_{xx} + u_{yy}, \qquad u(x,y,0) = \sin x \sin y$$

Tests standard 2D isotropic diffusion on $[0, 2\pi)^2$ ($N_x = N_y = 512$, $L_t = 2.0$, $N_t = 100$), rendering a contour-overlay animation and verifying against $u(x,y,t) = e^{-2t}\sin x \sin y$.

In [ ]:
# Define the symbols and the unknown function
t, x, y, kx, ky, omega = symbols('t x y kx ky omega')
u_func = Function('u') 
u = u_func(t, x, y)

# Define the 2D heat equation
eq = Eq(diff(u, t), diff(u, x, x) + diff(u, y, y))

# Initialize the solver
solver = PDESolver(eq)
Lt=2.0
# Configure the solver with the problem parameters
solver.setup(
    Lx=2 * np.pi, Ly=2 * np.pi,  # Spatial domain: [0, 2π] × [0, 2π]
    Nx=512, Ny=512,             # Spatial resolution: 512×512 points
    Lt=Lt, Nt=100,             # Total time: 1.0, number of time steps: 100
    initial_condition=lambda x, y: np.sin(x) * np.sin(y),  # Initial condition
    n_frames=50
)

# Solve the equation
solver.solve()

# Exact solution at t = 1
def u_exact(x, y, t):
    """
    Analytical solution of the 2D Poisson equation.
    The solution is given by u(x, y, t) = sin(x) * sin(y) * exp(-2t).
    """
    return np.sin(x) * np.sin(y) * np.exp(-2.0 * t)

# Automatic testing
n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='real') for i in range(n_test + 1)]

# Animate the solution
ani = solver.animate(overlay='contour', component='real')
HTML(ani.to_jshtml())

### With psiOp

$$u_t = \psi\mathrm{Op}\big(-(\xi^2 + \eta^2), u\big), \qquad u(x,y,0) = \sin x \sin y$$

Solves the same 2D heat equation by expressing the Laplacian via `psiOp(-(xi**2 + eta**2), u)` on $[0, 2\pi)^2$ ($N_x = N_y = 512$, $L_t = 2.0$, $N_t = 100$), verified against $u(x,y,t) = e^{-2t}\sin x \sin y$.

In [ ]:
# Define symbols
t, x, y, xi, eta = symbols('t x y xi eta', real=True)
u_func = Function('u')
u = u_func(t, x, y)

# Define the equation using psiOp
eq = Eq(diff(u, t), psiOp(-(xi**2 + eta**2), u))

# Create the solver instance
solver = PDESolver(eq)

# Total simulation time
Lt = 2.0

# Setup the solver
solver.setup(
    Lx=2 * np.pi, Ly=2 * np.pi,
    Nx=512, Ny=512,
    Lt=Lt, Nt=100,
    initial_condition=lambda x, y: np.sin(x) * np.sin(y),
    n_frames=50
)

# Solve the PDE
solver.solve()

# Define the exact solution
def u_exact(x, y, t):
    return np.sin(x) * np.sin(y) * np.exp(-2.0 * t)

# Test the result
n_test = 4
test_set = [
    solver.test(
        u_exact=u_exact,
        t_eval=i * Lt / n_test,
        threshold=1e-10,
        component='real'
    )
    for i in range(n_test + 1)
]
# Animate the solution
ani = solver.animate(overlay='front')
HTML(ani.to_jshtml())

## Schrödinger equation
### With PDE

$$i\,u_t = u_{xx} + u_{yy}, \qquad u(x,y,0) = e^{-(x^2 + y^2)} e^{i(x + y)}$$

Tests 2D free Schrödinger propagation and dispersion of a modulated Gaussian wave packet on $L_x = L_y = 20$ ($N_x = N_y = 256$, $L_t = 1.5$, $N_t = 200$) using `RK4`, compared against:

$$u(x,y,t) = \frac{1}{1 + 4it}\exp\!\left(i(x + y - 2t) - \frac{(x + 2t)^2 + (y + 2t)^2}{1 + 4it}\right)$$

In [ ]:
# Define the symbols and the unknown function
t, x, y, kx, ky, omega = symbols('t x y kx ky omega')
u_func = Function('u') 
u = u_func(t, x, y)

# 2D Schrödinger equation: i ∂t u = ∂xx u + ∂yy u
eq = Eq(I * diff(u, t), (diff(u, x, x) + diff(u, y, y)))

# Create the solver
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=1)

# Domain
Lx = Ly = 20
Nx = Ny = 256
Lt = 1.5
Nt = 200

# Initial condition: Gaussian wave packet modulated e^{-x^2 - y^2} e^{i(x + y)}
solver.setup(
    Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny, Lt=Lt, Nt=Nt,
    initial_condition=lambda x, y: np.exp(-x**2 - y**2) * np.exp(1j * (x + y))
)

# Solve the equation
solver.solve()

# Exact solution
def u_exact(x, y, t):
    """
    Analytical solution of the 2D Schrödinger equation.
    The solution is given by:
        u(x, y, t) = 1 / sqrt(1 + 4j * t)**2 * exp(i * (x + y - 2 * t)) * exp(-((x + 2t)^2 + (y + 2t)^2) / (1 + 4j * t)).
    """
    return 1 / np.sqrt(1 + 4j * t)**2 * np.exp(1j * (x + y - 2 * t)) * np.exp(-((x + 2 * t)**2 + (y + 2 * t)**2) / (1 + 4j * t))

# Automatic testing
n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=0.2, component='abs') for i in range(n_test + 1)]


### With psiOp

$$i\,u_t = \psi\mathrm{Op}\big(-(\xi^2 + \eta^2), u\big), \qquad u(x,y,0) = e^{-(x^2 + y^2)} e^{i(x + y)}$$

Solves the same 2D modulated Gaussian wave-packet problem as above using `psiOp(-(xi**2 + eta**2), u)` with the default solver settings on $L_x = L_y = 20$ ($N_x = N_y = 256$, $L_t = 1.5$).

In [ ]:
# Define the symbols and the unknown function
t, x, y, xi, eta = symbols('t x y xi eta', real=True)
u_func = Function('u') 
u = u_func(t, x, y)

# 2D Schrödinger equation: i ∂t u = ∂xx u + ∂yy u
eq = Eq(I * diff(u, t), psiOp(-(xi**2 + eta**2), u))

# Create the solver
solver = PDESolver(eq)

# Domain
Lx = Ly = 20
Nx = Ny = 256
Lt = 1.5
Nt = 200

# Initial condition: Gaussian wave packet modulated e^{-x^2 - y^2} e^{i(x + y)}
solver.setup(
    Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny, Lt=Lt, Nt=Nt,
    initial_condition=lambda x, y: np.exp(-x**2 - y**2) * np.exp(1j * (x + y))
)

# Solve the equation
solver.solve()

# Exact solution
def u_exact(x, y, t):
    """
    Analytical solution of the 2D Schrödinger equation.
    The solution is given by:
        u(x, y, t) = 1 / sqrt(1 + 4j * t)**2 * exp(i * (x + y - 2 * t)) * exp(-((x + 2t)^2 + (y + 2t)^2) / (1 + 4j * t)).
    """
    return 1 / np.sqrt(1 + 4j * t)**2 * np.exp(1j * (x + y - 2 * t)) * np.exp(-((x + 2 * t)**2 + (y + 2 * t)**2) / (1 + 4j * t))

# Automatic testing
n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=0.3, component='abs') for i in range(n_test + 1)]


### 2D free Schrödinger equation: anisotropic Gaussian wavepacket

$$i\,u_t + \tfrac12\,(u_{xx} + u_{yy}) = 0$$

The equation is separable, so the product of two 1D packets (different widths $\sigma_x$, $\sigma_y$ and momenta $k_x$, $k_y$) is an exact solution:

$$u(x,y,t) = g(x;\sigma_x,k_x,t)\;g(y;\sigma_y,k_y,t),\qquad g(s;\sigma,k,t) = \frac{1}{\sqrt{1 + it/\sigma^2}}\exp\!\left(-\frac{(s - kt)^2}{2\sigma^2(1 + it/\sigma^2)} + i\Big(ks - \frac{k^2 t}{2}\Big)\right)$$

It tests the complex-valued 2D path with a linear operator with a complex coefficient.

In [ ]:
# Define the symbols and the 2D free Schrodinger equation
t, x, y, kx, ky = symbols('t x y kx ky')
u_func = Function('u')
u = u_func(t, x, y)

eq = Eq(I * diff(u, t) + (diff(u, x, x) + diff(u, y, y)) / 2, 0)   # i u_t + (u_xx + u_yy)/2 = 0

# Wavepacket parameters
sx, sy = 1.2, 1.5        # Widths
kx0, ky0 = 0.6, -0.4     # Mean momenta

# Exact solution (product of two 1D packets)
def g(s, sigma, k, t):
    d = 1 + 1j * t / sigma**2
    return d**(-0.5) * np.exp(-(s - k * t)**2 / (2 * sigma**2 * d) + 1j * (k * s - k**2 * t / 2))

def u_exact(x, y, t):
    return g(x, sx, kx0, t) * g(y, sy, ky0, t)

# Initial condition (complex)
initial_condition = lambda x, y: u_exact(x, y, 0.0)

# Create the solver
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)

# Simulation parameters
Lx = Ly = 60
Nx = Ny = 256
Lt = 4.0
Nt = 200

# Setup
solver.setup(Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing: real and imaginary parts
n_test = 6
test_set_re = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-9, component='real') for i in range(n_test + 1)]
test_set_im = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-9, component='imag') for i in range(n_test + 1)]

## Fractional diffusion equation

$$u_t = -\mathrm{Op}\!\big((k_x^2 + k_y^2)^{\alpha/2}\big)\,u, \qquad \alpha = 1.5,\quad u(x,y,0) = \sin x \sin y$$

Tests 2D fractional diffusion $(-\Delta)^{\alpha/2}$ via Fourier multiplier `Op` on $[0, 2\pi)^2$ ($N_x = N_y = 128$, $L_t = 1.0$, $N_t = 100$) with dealiasing ratio $0.5$. For $(k_x, k_y) = (1, 1)$, the exact solution decays at rate $(1^2 + 1^2)^{\alpha/2} = 2^{3/4}$: $u(x,y,t) = \sin x \sin y\,e^{-2^{\alpha/2}t}$.

In [ ]:
# Define the symbols and the equation
t, x, y, kx, ky, omega = symbols('t x y kx ky omega')
u_func = Function('u') 
u = u_func(t, x, y)

# Parameters
alpha = 1.5  # Fractional order
nu = 0.01    # Classical diffusion coefficient

# 2D fractional diffusion equation
eq = Eq(diff(u, t), -Op((kx**2 + ky**2)**(alpha/2), u))

# Create the solver
solver = PDESolver(eq, dealiasing_ratio=0.5)

# Domain and initial condition setup
Lx, Ly = 2 * np.pi, 2 * np.pi  # Domain size
Nx, Ny = 128, 128             # Grid resolution
Lt = 1.0                      # Total simulation time
Nt = 100                     # Number of time steps

# Initial condition: sinusoidal wave in 2D
initial_condition = lambda x, y: np.sin(x) * np.sin(y)

# Setup the solver
solver.setup(
    Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny, Lt=Lt, Nt=Nt,
    initial_condition=initial_condition
)

# Solve the equation
solver.solve()

# Exact solution function at t = 1
def u_exact(x, y, t):
    return np.sin(x) * np.sin(y) * np.exp(-t * (1**2 + 1**2)**(alpha/2))

# Automatic testing
n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='real') for i in range(n_test + 1)]

## Non-linear equation

$$u_t = u_{xx} + u_{yy} + u^2, \qquad u(x,y,0) = e^{-(x^2 + y^2)}$$

Tests a 2D reaction–diffusion (Fujita-type) equation with quadratic nonlinearity $u^2$ on $L_x = L_y = 10$ ($N_x = N_y = 256$, $L_t = 1$, $N_t = 1000$), followed by an animation of the evolving profile and a short-time Gaussian heuristic comparison.

In [ ]:
# Symbolic variables
t, x, y = symbols('t x y')
u_func = Function('u')
u = u_func(t, x, y)

# Definition of the equation ∂u/∂t = Δu + u²
eq = Eq(diff(u, t), diff(u, x, 2) + diff(u, y, 2) + u**2)

# Creation of the solver
solver = PDESolver(eq)

# Square domain centered around 0
L = 10.0
N = 256
Lt = 1   # Reduced time for improved stability (nonlinear term)
Nt = 1000   # More time steps for precision

# Initial condition: t = 0
def initial_condition(x, y):
    return np.exp(-(x**2 + y**2))  # Centered Gaussian

# ❌ No simple exact solution for this nonlinear equation.
# For validation: use numerical convergence
# But if you absolutely want an "exact solution", here is an approximate case:
# This is a heuristic approximation, valid for short durations:
def u_exact(x, y, t):
    """Empirical approximation for short duration"""
    denom = 4 * t + 1
    return 1 / denom * np.exp(-(x**2 + y**2) / denom)

# Solver setup
solver.setup(
    Lx=L, Ly=L,
    Nx=N, Ny=N,
    Lt=Lt, Nt=Nt,
    initial_condition=initial_condition
)

# Solving
solver.solve()

# Validation at different times
n_test = 4
test_results = [
    solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1, component='real')
    for i in range(n_test + 1)
]

In [ ]:
ani = solver.animate(overlay='front', mode='imshow')
HTML(ani.to_jshtml())

## Wave equation
### With PDE

$$u_{tt} = u_{xx} + u_{yy}, \qquad u(x,y,0) = \sin x \sin y,\quad u_t(x,y,0) = 0$$

Tests the 2D scalar wave equation on $[0, 2\pi)^2$ ($N_x = N_y = 512$, $L_t = 2.0$, $N_t = 400$) using `RK4`, plotting energy conservation and verifying standing oscillation at frequency $\omega = \sqrt{k_x^2 + k_y^2} = \sqrt{2}$: $u(x,y,t) = \sin x \sin y \cos(\sqrt{2}\,t)$.

In [ ]:
# Define the symbols and the unknown function
t, x, y, kx, ky, omega = symbols('t x y kx ky omega')
u_func = Function('u') 
u = u_func(t, x, y)

# 2D wave equation
eq = Eq(diff(u, t, t), diff(u, x, x) + diff(u, y, y))

# Initialize the solver
solver = PDESolver(eq, time_scheme='RK4')

# Parameters
Lx = Ly = 2 * np.pi
Nx = Ny = 512
Lt = 2.0
Nt = 400

# Configure the solver with the problem parameters
solver.setup(
    Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny, Lt=Lt, Nt=Nt,
    initial_condition=lambda x, y: np.sin(x) * np.sin(y),
    initial_velocity=lambda x, y: np.zeros_like(x),
)

# Solve the equation
solver.solve()

solver.plot_energy() # (log=True) 

# Exact solution: sin(x) sin(y) cos(ωt), with ω = sqrt(2)
def u_exact(x, y, t):
    """
    Analytical solution of the 2D wave equation.
    The solution is given by u(x, y, t) = sin(x) * sin(y) * cos(ωt),
    where ω = sqrt(kx^2 + ky^2) = sqrt(1^2 + 1^2).
    """
    omega = np.sqrt(1**2 + 1**2)
    return np.sin(x) * np.sin(y) * np.cos(omega * t)

# Automatic testing
n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=5e-2, component='abs') for i in range(n_test + 1)]


### With a source term

$$u_{tt} = u_{xx} + u_{yy} - u + \cos x \cos y\big(5\cos(\sqrt{3}\,t) - \sqrt{3}\sin(\sqrt{3}\,t)\big)$$

Tests a forced 2D Klein–Gordon/wave-type equation with $u(x,y,0) = \cos x \cos y$ and $u_t(x,y,0) = 0$ on $[0, 4\pi)^2$ ($N_x = N_y = 256$, $L_t = 2.0$, $N_t = 400$) using `RK4` and $2/3$ dealiasing, verified against:

$$u(x,y,t) = \cos x \cos y\left[\left(1 + \frac{t}{2}\right)\cos(\sqrt{3}\,t) + \frac{\sqrt{3}}{6}(5t - 1)\sin(\sqrt{3}\,t)\right]$$

In [ ]:
# Define the symbols and the wave equation with a source term
t, x, y, omega = symbols('t x y omega')
u_func = Function('u')
u = u_func(t, x, y)

# Source term
source_term = cos(x) * cos(y) * (5 * cos(sqrt(3) * t) - sqrt(3) * sin(sqrt(3) * t))

# Equation with source term
eq = Eq(diff(u, t, t), diff(u, x, x) + diff(u, y, y) - u + source_term)

# Create the solver with ETD-RK4 scheme
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)

# Simulation parameters
Lt = 2.0  # Total simulation time
Lx, Ly = 4 * np.pi, 4 * np.pi  # Spatial domain size
Nx, Ny = 256, 256  # Number of grid points
Nt = 400  # Number of time steps

# Initial conditions
initial_condition = lambda x, y: np.cos(x) * np.cos(y)
initial_velocity = lambda x, y: np.zeros_like(x)

# Setup the solver
solver.setup(
    Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny, Lt=Lt, Nt=Nt,
    initial_condition=initial_condition,
    initial_velocity=initial_velocity
)

# Solve the equation
solver.solve()

# Plot energy evolution
solver.plot_energy() # (log=True)

# Exact solution at any time t
def u_exact(x, y, t):
    return np.cos(x) * np.cos(y) * ((1 + t/2) * np.cos(np.sqrt(3) * t) + (np.sqrt(3)/6) * (5*t - 1) * np.sin(np.sqrt(3) * t))

# Automatic testing at multiple times
n_test = 4
test_set = [
    solver.test(
        u_exact=u_exact,
        t_eval=i * Lt / n_test,
        threshold=1e-7,  # Adjust threshold if necessary
        component='real'
    )
    for i in range(n_test + 1)
]


In [ ]:
# Animate the solution
ani = solver.animate(overlay='contour')
HTML(ani.to_jshtml())

### With psiOp and source term

$$u_{tt} = \psi\mathrm{Op}\big(-(\xi^2 + \eta^2) - 1, u\big) + \cos x \cos y\big(5\cos(\sqrt{3}\,t) - \sqrt{3}\sin(\sqrt{3}\,t)\big)$$

Solves the same forced 2D equation as above using `psiOp` for the spatial operator $-(\xi^2 + \eta^2) - 1$ on $L_x = L_y = 20$ ($N_x = N_y = 256$, $L_t = 2.0$, $N_t = 200$).

In [ ]:
# Define the symbols and the wave equation with a source term
t, x, y, xi, eta, omega = symbols('t x y xi eta omega')
u_func = Function('u')
u = u_func(t, x, y)

# Source term
source_term = cos(x) * cos(y) * (5 * cos(sqrt(3) * t) - sqrt(3) * sin(sqrt(3) * t))

# Equation with source term
eq = Eq(diff(u, t, t), psiOp(-(xi**2 + eta**2) - 1, u) + source_term)

# Create the solver with ETD-RK4 scheme
solver = PDESolver(eq, dealiasing_ratio=2/3)

# Simulation parameters
Lt = 2.0  # Total simulation time
Lx, Ly = 20, 20  # Spatial domain size
Nx, Ny = 256, 256  # Number of grid points
Nt = 200  # Number of time steps

# Initial conditions
initial_condition = lambda x, y: np.cos(x) * np.cos(y)
initial_velocity = lambda x, y: np.zeros_like(x)

# Setup the solver
solver.setup(
    Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny, Lt=Lt, Nt=Nt,
    initial_condition=initial_condition,
    initial_velocity=initial_velocity
)

# Solve the equation
solver.solve()

# Plot energy evolution
solver.plot_energy() # (log=True)

# Exact solution at any time t
def u_exact(x, y, t):
    return np.cos(x) * np.cos(y) * ((1 + t/2) * np.cos(np.sqrt(3) * t) + (np.sqrt(3)/6) * (5*t - 1) * np.sin(np.sqrt(3) * t))

# Automatic testing at multiple times
n_test = 4
test_set = [
    solver.test(
        u_exact=u_exact,
        t_eval=i * Lt / n_test,
        threshold=0.6,  # Adjust threshold if necessary
        component='real'
    )
    for i in range(n_test + 1)
]


### With psiOp

$$u_{tt} = \psi\mathrm{Op}\big(-(\xi^2 + \eta^2), u\big), \qquad u(x,y,0) = \sin x \sin y,\quad u_t(x,y,0) = 0$$

Solves the homogeneous 2D wave equation on $[0, 2\pi)^2$ ($N_x = N_y = 512$, $L_t = 2.0$, $N_t = 400$) using `psiOp(-(xi**2 + eta**2), u)`, plotting energy conservation and verifying against $u(x,y,t) = \sin x \sin y \cos(\sqrt{2}\,t)$.

In [ ]:
# Define the symbols and the unknown function
t, x, y, xi, eta = symbols('t x y xi eta', real=True)
u_func = Function('u') 
u = u_func(t, x, y)

# 2D wave equation
eq = Eq(diff(u, t, t), psiOp(-(xi**2 + eta**2), u))

# Initialize the solver
solver = PDESolver(eq)

# Parameters
Lx = Ly = 2 * np.pi
Nx = Ny = 512
Lt = 2.0
Nt = 400

# Configure the solver with the problem parameters
solver.setup(
    Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny, Lt=Lt, Nt=Nt,
    initial_condition=lambda x, y: np.sin(x) * np.sin(y),
    initial_velocity=lambda x, y: np.zeros_like(x),
)

# Solve the equation
solver.solve()

solver.plot_energy() #(log=True) 

# Exact solution: sin(x) sin(y) cos(ωt), with ω = sqrt(2)
def u_exact(x, y, t):
    """
    Analytical solution of the 2D wave equation.
    The solution is given by u(x, y, t) = sin(x) * sin(y) * cos(ωt),
    where ω = sqrt(kx^2 + ky^2) = sqrt(1^2 + 1^2).
    """
    omega = np.sqrt(1**2 + 1**2)
    return np.sin(x) * np.sin(y) * np.cos(omega * t)

# Automatic testing
n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-4, component='abs') for i in range(n_test + 1)]


## Damped wave equation
### 2D Damped Wave Equation (Standard Derivatives)

$$u_{tt} + \gamma\,u_t - c^2\,(u_{xx} + u_{yy}) = 0, \qquad \gamma = 0.5,\ c = 1.0$$

$$u(x,y,0) = \exp\!\left(-\frac{x^2 + y^2}{2\sigma^2}\right), \qquad u_t(x,y,0) = 0, \qquad \sigma = 1.5$$

Tests the 2D damped wave (telegraph-type) equation expressed with standard partial derivatives using `time_scheme="RK4"` on a square domain of size $L_x = L_y = 40.0$ with grid resolution $N_x = N_y = 128$, over time horizon $L_t = 10.0$ with $N_t = 1000$ steps. Starting from a centered 2D Gaussian pulse at rest, the reference solution `u1_exact` is constructed spectrally via 2D FFT/IFFT using the damped dispersion frequency:

$$\omega_d = \sqrt{c^2(k_x^2 + k_y^2) - \frac{\gamma^2}{4}}$$

and validated at six equally spaced evaluation times (`t_eval = i * Lt / n_test` for `i = 1..6`) in relative norm against a `1e-2` threshold.


In [ ]:
# ==============================================================================
# Example 1: 2D Damped Wave Equation (Standard Derivatives)
# Equation: u_tt + 0.5*u_t = 1.0*(u_xx + u_yy)
# ==============================================================================
print("\n--- Example 1: 2D Damped Wave (Standard Derivatives) ---")

# 1. Define symbols and the equation
t, x, y = symbols('t x y')
u = Function('u')(t, x, y)

gam, c = 0.5, 1.0
eq1 = Eq(diff(u, t, 2) + gam * diff(u, t) - c**2 * (diff(u, x, 2) + diff(u, y, 2)), 0)
solver1 = PDESolver(eq1, time_scheme="RK4")

# 2. Domain and parameters
Lx, Nx = 40.0, 128
Ly, Ny = Lx, Nx  # Square domain
Lt, Nt = 10.0, 1000
sig = 1.5

# 3. Exact solution (via spectral evolution of the Gaussian initial condition)
def u1_exact(X, Y, T):
    kx = 2 * np.pi * np.fft.fftfreq(X.shape[0], Lx / Nx)
    ky = 2 * np.pi * np.fft.fftfreq(X.shape[1], Lx / Nx)
    KX, KY = np.meshgrid(kx, ky, indexing='ij')
    wd = np.sqrt(c**2 * (KX**2 + KY**2) - gam**2 / 4 + 0j)
    Sx = np.where(np.abs(wd * T) < 1e-9, T, np.sin(wd * T) / np.where(wd == 0, 1, wd))
    u0_vals = np.exp(-(X**2 + Y**2) / (2 * sig**2))
    return np.real(np.fft.ifft2(np.fft.fft2(u0_vals) * np.exp(-gam * T / 2) * (np.cos(wd * T) + gam / 2 * Sx)))

# 4. Setup
solver1.setup(
    Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny, Lt=Lt, Nt=Nt,
    initial_condition=lambda X, Y: np.exp(-(X**2 + Y**2) / (2 * sig**2)),
    initial_velocity=lambda X, Y: np.zeros_like(X),
    plot=True
)

# 5. Solving
solver1.solve()

# 6. Automatic testing
n_test = 6
test_set1 = [
    solver1.test(
        u_exact=u1_exact, 
        t_eval=i * Lt / n_test, 
        threshold=1e-2, 
        component='real', 
        norm='relative'
    )
    for i in range(1, n_test + 1)
]


### 2D Damped Wave with Pseudo-Differential Operator (psiOp)

$$u_{tt} + \gamma\,u_t = \psi\mathrm{Op}\big(-c^2(\xi^2 + \eta^2),\,u\big), \qquad \gamma = 0.5,\ c = 1.0$$

Solves the same physical 2D damped Gaussian wave-pulse problem as Example 1, but reformulates the spatial Laplacian as a pseudo-differential operator `psiOp(-c**2 * (xi**2 + eta**2), up)` with `time_scheme="default"`, `dealiasing_ratio=2/3`, `boundary_condition='nonperiodic'`, and refined temporal resolution (`Nt_psi = 4000` over $L_t = 10.0$ on the same $40.0 \times 40.0$, $128 \times 128$ grid). Because nonperiodic boundary handling clips the field near the domain edges as the wave spreads, the relative error test across `i = 1..6` uses a relaxed threshold of `1e-1` compared against the same spectral reference solution `u2_exact`.

In [ ]:
# ==============================================================================
# Example 2: 2D Damped Wave with Pseudo-Differential Operator (psiOp)
# Equation: u_tt + 0.5*u_t = psiOp(-1.0*(xi^2 + eta^2), u)
# ==============================================================================
print("\n--- Example 2: 2D Damped Wave with psiOp (Nonperiodic) ---")

# 1. Define symbols and the equation
ts, xs, ys, xi, eta = symbols('t x y xi eta', real=True)
up = Function('u')(ts, xs, ys)

eq2 = Eq(diff(up, ts, 2) + gam * diff(up, ts), psiOp(-c**2 * (xi**2 + eta**2), up))
solver2 = PDESolver(eq2, time_scheme="default", dealiasing_ratio=2/3)

# 2. Domain and parameters (reusing Lx, Nx, Lt, sig from above)
# Higher Nt is often beneficial for psiOp / nonperiodic combinations
Nt_psi = 4000  

# 3. Exact solution (same physical problem, so we reuse the exact solution function)
def u2_exact(X, Y, T):
    kx = 2 * np.pi * np.fft.fftfreq(X.shape[0], Lx / Nx)
    ky = 2 * np.pi * np.fft.fftfreq(X.shape[1], Lx / Nx)
    KX, KY = np.meshgrid(kx, ky, indexing='ij')
    wd = np.sqrt(c**2 * (KX**2 + KY**2) - gam**2 / 4 + 0j)
    Sx = np.where(np.abs(wd * T) < 1e-9, T, np.sin(wd * T) / np.where(wd == 0, 1, wd))
    u0_vals = np.exp(-(X**2 + Y**2) / (2 * sig**2))
    return np.real(np.fft.ifft2(np.fft.fft2(u0_vals) * np.exp(-gam * T / 2) * (np.cos(wd * T) + gam / 2 * Sx)))

# 4. Setup
solver2.setup(
    Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny, Lt=Lt, Nt=Nt_psi,
    boundary_condition='nonperiodic',
    initial_condition=lambda X, Y: np.exp(-(X**2 + Y**2) / (2 * sig**2)),
    initial_velocity=lambda X, Y: np.zeros_like(X),
    plot=True
)

# 5. Solving
solver2.solve()

# 6. Automatic testing
# Note: threshold is slightly relaxed (1e-1) because 'nonperiodic' mode uses 
# heuristic boundary clipping, which introduces small errors when the wave 
# reaches the domain edges.
n_test = 6
test_set2 = [
    solver2.test(
        u_exact=u2_exact, 
        t_eval=i * Lt / n_test, 
        threshold=1e-1, 
        component='real', 
        norm='relative'
    )
    for i in range(1, n_test + 1)
]

## Klein-Gordon equation

$$u_{tt} = c^2(u_{xx} + u_{yy}) - m^2 u, \qquad c = 1.0,\ m = 1.0$$

Tests the 2D linear Klein–Gordon equation on $[0, 2\pi)^2$ ($N_x = N_y = 512$, $L_t = 2.0$, $N_t = 200$) with $u(x,y,0) = \sin x \sin y$ and $u_t(x,y,0) = 0$. For $(k_x, k_y) = (1, 1)$, the dispersion relation gives $\omega = \sqrt{c^2(k_x^2 + k_y^2) + m^2} = \sqrt{3}$ and exact solution $u(x,y,t) = \sin x \sin y \cos(\omega t)$.

In [ ]:
# Physical parameters
c = 1.0  # Wave speed
m = 1.0  # Field mass

# Definition of the 2D Klein-Gordon equation
t, x, y, kx, ky, omega = symbols('t x y kx ky omega')
u_func = Function('u') 
u = u_func(t, x, y)

klein_gordon_eq = Eq(diff(u, t, t), c**2*(diff(u, x, x) + diff(u, y, y)) - m**2*u)

# Creation of the solver 
solver = PDESolver(klein_gordon_eq)

# Domain configuration
L = 2 * np.pi  # Square domain [0, 2π] × [0, 2π]
N = 512        # Grid points per dimension
Lt = 2.0  # Final time
Nt = 200       # Time steps

# Initial conditions
kx = 1
ky = 1
omega = np.sqrt(c**2*(kx**2 + ky**2) + m**2)  # Direct numerical calculation

solver.setup(
    Lx=L, Ly=L,
    Nx=N, Ny=N,
    Lt=Lt, Nt=Nt,
    initial_condition=lambda x, y: np.sin(x) * np.sin(y),
    initial_velocity=lambda x, y: np.zeros_like(x)  # Initial time derivative is zero
)

# Replace the definition of the exact solution with:
omega_val = float(np.sqrt(c**2*(kx**2 + ky**2) + m**2))  # Convert to float

def u_exact(x, y, t):
    return np.sin(x) * np.sin(y) * np.cos(omega_val * t)

# Solving and validation
solver.solve()

solver.plot_energy() #(log=True) 

# Automatic testing
n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='real') for i in range(n_test + 1)]

## Biharmonic equation

$$u_t = -(u_{xxxx} + 2u_{xxyy} + u_{yyyy}) = -\Delta^2 u, \qquad u(x,y,0) = \sin x \sin y$$

Tests fourth-order biharmonic dissipation on $[0, 2\pi)^2$ ($N_x = N_y = 512$, $L_t = 2.0$, $N_t = 400$) with periodic boundary conditions. Since $\Delta^2(\sin x \sin y) = (1^2 + 1^2)^2 \sin x \sin y = 4\sin x \sin y$, the exact solution is $u(x,y,t) = e^{-4t}\sin x \sin y$.

In [ ]:
# Definition of the 2D biharmonic equation
t, x, y, kx, ky, omega = symbols('t x y kx ky omega')
u_func = Function('u') 
u = u_func(t, x, y)

biharmonic_eq = Eq(diff(u, t), -(diff(u, x, 4) + 2*diff(u, x, 2, y, 2) + diff(u, y, 4)))

# Creation of the solver with periodic boundary conditions
solver = PDESolver(biharmonic_eq)

# Configuration of the domain
L = 2 * np.pi  # Square domain [0, 2π] × [0, 2π]
N = 512         # Grid points per dimension
Lt = 2.0  # Final time
Nt = 400       # Time steps

# Initial sinusoidal 2D condition
initial_condition = lambda x, y: np.sin(x) * np.sin(y)

solver.setup(
    Lx=L, Ly=L,
    Nx=N, Ny=N,
    Lt=Lt, Nt=Nt,
    initial_condition=initial_condition
)

# Corresponding exact solution
def u_exact(x, y, t):
    return np.sin(x) * np.sin(y) * np.exp(-4*t)

# Solving 
solver.solve()

# Automatic testing
n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='real') for i in range(n_test + 1)]

## Burgers' equation
### solution constant along the diagonal

$$u_t + u\,(u_x + u_y) = \nu\,(u_{xx} + u_{yy})$$

For $u(x,y,t) = \tfrac12 V(x+y,\,t)$, $V$ solves the 1D Burgers equation with viscosity $2\nu$, hence the periodic Cole–Hopf solution

$$u(x,y,t) = \frac{2\nu a\,e^{-2\nu t}\sin(x+y)}{1 + a\,e^{-2\nu t}\cos(x+y)}$$

(period $2\pi$ in both $x$ and $y$). It exercises the 2D nonlinear path with $u_x$ and $u_y$.

In [ ]:
# Define the symbols and the 2D Burgers equation
t, x, y, kx, ky = symbols('t x y kx ky')
u_func = Function('u')
u = u_func(t, x, y)

nu = 0.1
eq = Eq(diff(u, t) + u * (diff(u, x) + diff(u, y)) - nu * (diff(u, x, x) + diff(u, y, y)), 0)
# u_t + u (u_x + u_y) = nu (u_xx + u_yy)

# Exact solution (diagonal Cole-Hopf)
a = 0.8
def u_exact(x, y, t):
    E = np.exp(-2 * nu * t)
    return 2 * nu * a * E * np.sin(x + y) / (1 + a * E * np.cos(x + y))

# Initial condition
initial_condition = lambda x, y: u_exact(x, y, 0.0)

# Create the solver
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)

# Simulation parameters
Lx = Ly = 2 * np.pi
Nx = Ny = 64
Lt = 2.0
Nt = 200

# Setup
solver.setup(Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing
n_test = 6
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-7, component='real') for i in range(n_test + 1)]

## Method of manufactured solutions

Pick any smooth $u^\star(x,t)$, let sympy compute the source $f$ it requires, and add $f$ to the equation: $u^\star$ is then the exact solution, whatever the nonlinearity. This is the most general way to exercise the nonlinear terms.

1. **1D**: a coefficient on $u_t$, explicit time dependence ($\sin t\,u^2$) and a convective term.
2. **2D**: a mixed derivative, $u\,u_{xy}$.

### 2D: $u_t = \nu\,\Delta u - u\,u_{xy} + f$

The term $u\,u_{xy}$ acts as an anti-diffusion in the $xy$ direction: the problem is well posed only if $|u| < 2\nu$, hence $\nu = 0.2$ and an amplitude of $0.3$.

In [ ]:
# Define the symbols and the equation
t, x, y, kx, ky = symbols('t x y kx ky')
u_func = Function('u')
u = u_func(t, x, y)

nu = 0.2

# Manufactured solution and the source it requires (SymPy uniquement pour f et eq)
u_star = 0.3 * exp(-t) * sin(x) * sin(y)
f = diff(u_star, t) - nu * (diff(u_star, x, x) + diff(u_star, y, y)) + u_star * diff(u_star, x, y)

eq = Eq(diff(u, t) - nu * (diff(u, x, x) + diff(u, y, y)) + u * diff(u, x, y) - f, 0)

# Exact solution & initial condition en NumPy pur (sans lambdify)
def u_exact(x, y, t):
    return 0.3 * np.exp(-t) * np.sin(x) * np.sin(y)

initial_condition = lambda x, y: 0.3 * np.sin(x) * np.sin(y)

# Create the solver
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)

# Simulation parameters
Lx = Ly = 2 * np.pi
Nx = Ny = 128
Lt = 2.0
Nt = 200

# Setup
solver.setup(Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing
n_test = 6
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-9, component='real') for i in range(n_test + 1)]